In [1]:
!pip install bert-score groq pandas tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.3 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
import pandas as pd
from tqdm import tqdm
from bert_score import score as bert_score
from groq import Groq

os.environ["GROQ_API_KEY"] = ""  # <-- paste key
client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [3]:


df = pd.read_csv("/content/adapted_texts_119.csv")
df = df[df["adapted_text"] != "[ADAPTATION_FAILED]"]
print(f"Loaded {len(df)} rows")
print(df.groupby("target_region").size())
df.head(3)

Loaded 119 rows
target_region
Kerala         19
Maharashtra    20
Punjab         20
Rajasthan      19
Tamil Nadu     21
West Bengal    20
dtype: int64


,story_id,source_culture,source_text,target_region,target_language,adapted_text
0,0,Australian,(approx. 250-300 words)\n \n Title: The Littl...,Tamil Nadu,Tamil,The Little Kuravan's Big Mistake\n\nOnce upon ...
1,0,Australian,(approx. 250-300 words)\n \n Title: The Littl...,Punjab,Punjabi,The Little Farmer's Big Mistake\n\nOnce upon a...
2,0,Australian,(approx. 250-300 words)\n \n Title: The Littl...,West Bengal,Bengali,The Little Monk's Big Mistake\n\nOnce upon a t...


In [ ]:
# ── Cell 4: METRIC 1 — BERTScore (semantic similarity)
# Measures: does the adapted text preserve the meaning of the source?
# Score close to 1.0 = meaning preserved
# Score too low = plot was lost during adaptation

print("Computing BERTScore (takes 2-3 mins)...")

sources  = df["source_text"].tolist()
adapted  = df["adapted_text"].tolist()

P, R, F1 = bert_score(adapted, sources, lang="en", verbose=False)

df["bertscore_precision"] = P.numpy()
df["bertscore_recall"]    = R.numpy()
df["bertscore_f1"]        = F1.numpy()

print("Done!")
print(f"\nMean BERTScore F1: {df['bertscore_f1'].mean():.4f}")
print(f"\nBy region:")
print(df.groupby("target_region")["bertscore_f1"].mean().round(4))

Computing BERTScore (takes 2-3 mins)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# ── Cell 5: METRIC 2 — CSI Replacement Rate ───────────────────────────────────
# Measures: how many Western cultural items were actually replaced?
# We use a hardcoded Western CSI list (food, festivals, sports, places, brands)
# In a full project you'd use the DIWALI lexicon for Indian CSIs

WESTERN_CSIS = [
    # Food
    "turkey", "hamburger", "hot dog", "pizza", "bagel", "pancake",
    "bacon", "donut", "burrito", "sandwich", "steak", "mac and cheese",
    # Drinks
    "beer", "whiskey", "wine", "soda", "coffee",
    # Festivals / events
    "thanksgiving", "halloween", "christmas", "super bowl", "fourth of july",
    "black friday", "easter", "new year's eve",
    # Sports
    "football", "baseball", "basketball", "nfl", "nba", "mlb",
    # Brands / places
    "walmart", "mcdonald's", "starbucks", "target", "amazon",
    "new york", "los angeles", "chicago", "texas",
    # Transport
    "pickup truck", "school bus", "subway",
    # Names
    "john", "mike", "jake", "sarah", "emily", "chris",
]

def count_western_csis(text):
    text_lower = text.lower()
    return [csi for csi in WESTERN_CSIS if csi in text_lower]

def csi_replacement_rate(source, adapted):
    source_csis  = count_western_csis(source)
    adapted_csis = count_western_csis(adapted)
    if not source_csis:
        return None, [], []
    replaced = [c for c in source_csis if c not in adapted_csis]
    rate = len(replaced) / len(source_csis)
    return rate, source_csis, replaced

rates, found_csis, replaced_csis = [], [], []
for _, row in df.iterrows():
    rate, found, replaced = csi_replacement_rate(
        row["source_text"], row["adapted_text"]
    )
    rates.append(rate)
    found_csis.append(found)
    replaced_csis.append(replaced)

df["csi_replacement_rate"] = rates
df["csis_found"]            = [str(x) for x in found_csis]
df["csis_replaced"]         = [str(x) for x in replaced_csis]

# Drop rows where no CSIs were found in source
df_csi = df[df["csi_replacement_rate"].notna()]

print(f"Stories with at least 1 Western CSI: {len(df_csi)}")
print(f"\nMean CSI replacement rate: {df_csi['csi_replacement_rate'].mean():.4f}")
print(f"\nBy region:")
print(df_csi.groupby("target_region")["csi_replacement_rate"].mean().round(4))

Stories with at least 1 Western CSI: 30

Mean CSI replacement rate: 1.0000

By region:
target_region
Kerala         1.0
Maharashtra    1.0
Punjab         1.0
Rajasthan      1.0
Tamil Nadu     1.0
West Bengal    1.0
Name: csi_replacement_rate, dtype: float64


In [ ]:
# ── METRIC 2 — CSI Replacement Rate ──────────────────────────────────────────
import re
from collections import Counter

WESTERN_CSIS = [
    # Food — American
    "turkey", "hamburger", "hot dog", "pizza", "bagel", "pancake",
    "bacon", "donut", "burrito", "steak", "mac and cheese",
    "biscuits and gravy", "clam chowder",
    # Food — British / Australian
    "fish and chips", "meat pie", "vegemite", "sausage roll", "scones",
    # Drinks
    "beer", "whiskey", "wine", "rootbeer", "eggnog",
    # Festivals
    "thanksgiving", "halloween", "christmas", "super bowl",
    "fourth of july", "black friday", "easter",
    "bonfire night", "guy fawkes", "boxing day",
    "australia day", "anzac day",
    # Sports — cricket intentionally excluded (used in Indian adaptations too)
    "american football", "baseball", "nfl", "nba", "mlb", "nhl",
    "rugby", "afl", "footy",
    # Brands & places
    "walmart", "mcdonald's", "starbucks",
    "new york", "los angeles", "chicago",
    "london", "oxford", "buckingham palace",
    "sydney", "melbourne", "outback",
    # Animals specific to Western source cultures
    "koala", "kangaroo", "wombat", "wallaby",
    # Transport
    "pickup truck", "school bus",
    # Names — full combos only to avoid substring collisions
    "john smith", "jack and jill",
]

def count_western_csis(text):
    text_lower = text.lower()
    found = []
    for csi in WESTERN_CSIS:
        pattern = r"\b" + re.escape(csi) + r"\b"
        if re.search(pattern, text_lower):
            found.append(csi)
    return found

def csi_replacement_rate(source, adapted):
    clean_source = source[:2000]
    source_csis  = count_western_csis(clean_source)
    adapted_csis = count_western_csis(adapted)
    if not source_csis:
        return None, [], [], []
    replaced = [c for c in source_csis if c not in adapted_csis]
    retained = [c for c in source_csis if c in adapted_csis]
    rate = len(replaced) / len(source_csis)
    return rate, source_csis, replaced, retained

rates, found_csis, replaced_csis, retained_csis = [], [], [], []
for _, row in df.iterrows():
    rate, found, replaced, retained = csi_replacement_rate(
        row["source_text"], row["adapted_text"]
    )
    rates.append(rate)
    found_csis.append(found)
    replaced_csis.append(replaced)
    retained_csis.append(retained)

df["csi_replacement_rate"] = rates
df["csis_found"]            = [str(x) for x in found_csis]
df["csis_replaced"]         = [str(x) for x in replaced_csis]
df["csis_retained"]         = [str(x) for x in retained_csis]

df_csi = df[df["csi_replacement_rate"].notna()]

print(f"Stories with >=1 Western CSI in source : {len(df_csi)} / {len(df)}")
print(f"Mean CSI replacement rate              : {df_csi['csi_replacement_rate'].mean():.4f}")
print(f"\nBy region:")
print(df_csi.groupby("target_region")["csi_replacement_rate"].mean().round(4).to_string())

print(f"\nCSIs detected across source texts:")
all_found = [c for row in found_csis if row for c in row]
for csi, count in Counter(all_found).most_common():
    print(f"  '{csi}' x{count}")

all_retained = [c for row in retained_csis if row for c in row]
if all_retained:
    print(f"\nCSIs NOT replaced (retained in adapted text):")
    for csi, count in Counter(all_retained).most_common():
        print(f"  '{csi}' x{count}  <- investigate these")
else:
    print(f"\nAll detected CSIs were successfully replaced.")

print(f"\nNote: {len(df) - len(df_csi)}/{len(df)} stories had no detectable Western CSIs.")
print("This reflects the fable genre, not a pipeline failure.")

Stories with >=1 Western CSI in source : 24 / 119
Mean CSI replacement rate              : 1.0000

By region:
target_region
Kerala         1.0
Maharashtra    1.0
Punjab         1.0
Rajasthan      1.0
Tamil Nadu     1.0
West Bengal    1.0

CSIs detected across source texts:
  'koala' x12
  'wallaby' x12
  'outback' x12
  'wombat' x6
  'kangaroo' x6

All detected CSIs were successfully replaced.

Note: 95/119 stories had no detectable Western CSIs.
This reflects the fable genre, not a pipeline failure.


In [ ]:
# ── Cell 6: METRIC 3 — LLM-as-Judge (Cultural Depth Score) ───────────────────
# Measures: does the adapted text reflect the VALUES and NORMS of the region?
# Not just word swaps — does it feel culturally authentic?
# Score: 1-5 (1=generic, 5=deeply authentic)
# Uses Groq API — runs ~120 calls so will use ~15k tokens

JUDGE_PROMPT = """You are an expert in Indian regional cultures.
Evaluate how well the following adapted story reflects the culture of {region}, India.

Score on a scale of 1 to 5:
1 = Generic / no cultural adaptation visible
2 = Superficial — only a few words changed
3 = Moderate — some cultural elements present but feels generic
4 = Good — culturally specific, mostly authentic
5 = Excellent — deeply reflects {region} values, norms, and daily life

Respond with ONLY a JSON object like this:
{{"score": 4, "reason": "one sentence explanation"}}

Adapted story:
{adapted_text}"""

def llm_judge(adapted_text, region, max_retries=3):
    prompt = JUDGE_PROMPT.format(region=region, adapted_text=adapted_text[:800])
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=100,
            )
            raw = resp.choices[0].message.content.strip()
            # Clean up response
            raw = raw.replace("json", "").replace("", "").strip()
            parsed = json.loads(raw)
            return parsed.get("score"), parsed.get("reason", "")
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return None, "failed"

scores, reasons = [], []
for _, row in tqdm(df.iterrows(), total=len(df), desc="LLM Judge"):
    score, reason = llm_judge(row["adapted_text"], row["target_region"])
    scores.append(score)
    reasons.append(reason)
    time.sleep(3)  # conservative to avoid rate limit

df["cultural_depth_score"] = scores
df["cultural_depth_reason"] = reasons

print(f"\nMean Cultural Depth Score: {df['cultural_depth_score'].mean():.2f} / 5")
print(f"\nBy region:")
print(df.groupby("target_region")["cultural_depth_score"].mean().round(2))

LLM Judge: 100%|██████████| 119/119 [06:44<00:00,  3.40s/it]


Mean Cultural Depth Score: 3.92 / 5

By region:
target_region
Kerala         3.95
Maharashtra    3.90
Punjab         3.90
Rajasthan      3.89
Tamil Nadu     3.95
West Bengal    3.95
Name: cultural_depth_score, dtype: float64


In [ ]:
# ── Cell 7: Combined results summary ─────────────────────────────────────────
summary = df.groupby("target_region").agg(
    bertscore_f1        = ("bertscore_f1",        "mean"),
    csi_replacement_rate= ("csi_replacement_rate", "mean"),
    cultural_depth_score= ("cultural_depth_score", "mean"),
    count               = ("story_id",             "count"),
).round(3)

print("=" * 60)
print("EVALUATION RESULTS BY REGION")
print("=" * 60)
print(summary.to_string())
print("\nHigher BERTScore = meaning better preserved")
print("Higher CSI Rate  = more Western items replaced")
print("Higher Depth     = more culturally authentic (1-5)")

EVALUATION RESULTS BY REGION
               bertscore_f1  csi_replacement_rate  cultural_depth_score  count
target_region                                                                 
Kerala                0.901                   1.0                 3.947     19
Maharashtra           0.916                   1.0                 3.900     20
Punjab                0.912                   1.0                 3.900     20
Rajasthan             0.903                   1.0                 3.895     19
Tamil Nadu            0.908                   1.0                 3.952     21
West Bengal           0.910                   1.0                 3.950     20

Higher BERTScore = meaning better preserved
Higher CSI Rate  = more Western items replaced
Higher Depth     = more culturally authentic (1-5)


In [ ]:
df.to_csv("eval_results.csv", index=False)
summary.to_csv("eval_summary.csv")
print("Saved eval_results.csv and eval_summary.csv")

from google.colab import files
files.download("eval_results.csv")
files.download("eval_summary.csv")

Saved eval_results.csv and eval_summary.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>